<a id="top"></a>

<div style="border:3px solid #2ca02c;background:#eaffea;padding:18px 22px;border-radius:12px;margin:10px 0 18px 0;">
  <div style="font-size:44px;font-weight:900;line-height:1.1;">Analyse von Schlaf- und Lebensstilfaktoren</div>
  <div style="margin-top:12px;display:flex;gap:10px;flex-wrap:wrap;">
    <a href="./01-EDA-Hypothesenentwicklung.ipynb" style="display:inline-block;padding:8px 16px;border-radius:999px;background:#fff;color:#1b5e20;font-size:18px;font-weight:700;text-decoration:none;border:1px solid rgba(46,125,50,0.45);">Explorative Datenanalyse</a>
    <a href="./02-Feature-Engineering-Clustering-Modeling-MapReduce.ipynb" style="display:inline-block;padding:8px 16px;border-radius:999px;background:#fff;color:#1b5e20;font-size:18px;font-weight:700;text-decoration:none;border:1px solid rgba(46,125,50,0.45);">Feature Engineering &amp; Modeling</a>
    <span style="display:inline-block;padding:8px 16px;border-radius:999px;background:#2e7d32;color:#fff;font-size:18px;font-weight:800;box-shadow:0 2px 6px rgba(0,0,0,0.10);">Evaluation &amp; Deployment</span>
  </div>
</div>


<div style="margin:26px 0 14px 0;">
  <div style="font-size:14px;letter-spacing:1px;color:#555;text-transform:uppercase;">
    Notebook 3
  </div>
  <div style="font-size:30px;font-weight:700;margin-top:4px;">
    Evaluation &amp; Deployment
  </div>
</div>

In [1]:
# Import der Ergebnisse
import pandas as pd
import numpy as np

import plotly.express as px
import plotly.graph_objects as go

from IPython.display import display
from plotly.subplots import make_subplots

from typing import cast

px.defaults.template = "simple_white"


labels = {
    "age": "Age",
    "stress": "Stress Level (1–10)",
    "sleepDuration": "Sleep Duration (hours)",
    "sleepQuality": "Sleep Quality (1–10)",
    "heartRate": "Heart Rate (bpm)",
    "activity": "Physical Activity (min/day)",
    "alcohol": "Alcohol (drinks/week)",
    "sugar": "Sugar Intake (g/day)",
    "steps": "Daily Steps",
    "shortSleep": "Short Sleep",
    "lifestyleRisk": "Lifestyle Risk",
    "stressSleepMismatch": "Stress–Sleep Mismatch"
}

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.4f}".format)

model_compare = pd.read_csv("data/regression_results.csv")
clf_compare = pd.read_csv("data/classification_results.csv")

perf = pd.read_csv("data/model_performance_summary.csv")

cv_scores_lr = pd.read_csv("data/cv_scores_lr.csv").squeeze("columns")
cv_scores_reg = pd.read_csv("data/cv_scores_regression.csv").squeeze("columns")
cv_scores_clf = pd.read_csv("data/cv_scores_classification.csv").squeeze("columns")

feat_reg = pd.read_csv("data/feature_importance_rf.csv")
#feat_reg["feature"] = feat_reg["feature"].map(lambda c: labels.get(c, c))
feat_reg["feature"] = feat_reg["feature"].map(lambda c: labels.get(str(c), c))
feat_reg = feat_reg.sort_values("importance")

cluster_data = pd.read_csv("data/cluster_data.csv")
cluster_summary = pd.read_csv("data/cluster_summary.csv")

print("Loaded artifacts:")
print("Model Performance:", perf.shape)
print("Regression Compare:", model_compare.shape)
print("Classification Compare:", clf_compare.shape)
print("CV Regression:", cv_scores_reg.shape)
print("CV Classification:", cv_scores_clf.shape)
print("Feature Importance:", feat_reg.shape)
print("Cluster Data:", cluster_data.shape)
print("Cluster Summary:", cluster_summary.shape)

Loaded artifacts:
Model Performance: (6, 2)
Regression Compare: (2, 4)
Classification Compare: (2, 6)
CV Regression: (5,)
CV Classification: (5,)
Feature Importance: (7, 2)
Cluster Data: (2000, 4)
Cluster Summary: (3, 5)


## 1. Ziel dieses Notebooks

Dieses Notebook bildet den **abschließenden Teil der Analysepipeline**.  
Nachdem im vorherigen Notebook prädiktive Modelle entwickelt wurden, liegt der Fokus nun auf der **inhaltlichen Interpretation der Ergebnisse** und der **Überprüfung datenbasierter Hypothesen**.

Ziel ist es, aus den modellierten Zusammenhängen **interpretierbare Erkenntnisse über Lifestyle-Faktoren, Schlafqualität und Stressniveau** abzuleiten.

**Einordnung in die Projektpipeline**

Die Analyse des Datensatzes erfolgte in drei aufeinander aufbauenden Notebooks:

| Notebook | Fokus | Inhalte |
|--------|------|------|
| **Notebook 1** | Datenverständnis & Exploration | Datenstruktur, Verteilungen, Zusammenhänge, erste Hypothesen |
| **Notebook 2** | Feature Engineering & Modellierung | neue Features, Clustering, Regressions- und Klassifikationsmodelle |
| **Notebook 3** | Evaluation & Interpretation | Modellinterpretation, Hypothesenprüfung, Erkenntnisse |

Dieses Notebook nutzt daher **die bereits trainierten Modelle und erzeugten Features**, um deren Bedeutung für die zugrunde liegenden Fragestellungen zu analysieren.

Die Analyse konzentriert sich auf drei zentrale Themenbereiche:

##### 1) Schlafqualität

Welche Faktoren beeinflussen die Schlafqualität am stärksten?
- der Einfluss von Stress
- der Einfluss körperlicher Aktivität
- mögliche Lifestyle-Risikofaktoren

Die Analyse basiert auf dem im vorherigen Notebook entwickelten **Regressionsmodell zur Vorhersage der Schlafqualität**.

##### 2) Stressrisiko

Welche Faktoren erhöhen die Wahrscheinlichkeit eines hohen Stresslevels?

Hierbei wird das entwickelte **Klassifikationsmodell für High-Stress-Fälle** verwendet.

Zentrale Fragestellungen:

- Welche Lifestyle-Faktoren begünstigen hohen Stress?
- Welche Faktoren wirken stabilisierend?

##### 3) Lifestyle-Segmente

Neben individuellen Einflussfaktoren werden auch **typische Lifestyle-Profile** betrachtet.

Hierzu werden die im vorherigen Notebook identifizierten **Cluster von Lebensstilen** analysiert.

Ziel ist es zu prüfen:

- ob bestimmte Lifestyle-Typen häufiger unter Stress leiden
- ob Cluster systematisch unterschiedliche Schlafqualitäten aufweisen

Die Interpretation der Modelle erfolgt in drei Schritten:

1. **Zusammenfassung der Modellperformance**
2. **Analyse der wichtigsten Einflussfaktoren**
3. **Prüfung der zuvor formulierten Hypothesen**

Am Ende dieses Notebooks sollen drei Fragen beantwortet werden:

- **Welche Faktoren bestimmen Schlafqualität und Stressniveau?**
- **Welche Lifestyle-Typen lassen sich im Datensatz erkennen?**
- **Welche Hypothesen aus der explorativen Analyse werden bestätigt?**

Die Ergebnisse bilden die Grundlage für eine abschließende Bewertung der Analyse und mögliche weiterführende Untersuchungen.

## 2. Überblick über die Modellresultate

Bevor die inhaltliche Interpretation beginnt, werden die wichtigsten Ergebnisse
der Modellierung kurz zusammengefasst. 

Ziel ist es,

- die Modellleistung kurz einzuordnen,
- das leistungsstärkste Modell zu identifizieren
- sowie mögliche Hinweise auf Overfitting zu prüfen.

Dabei werden sowohl **Test-Performance** als auch **Cross-Validation-Ergebnisse**
betrachtet.

Im vorherigen Notebook wurden zwei Modelltypen entwickelt:

| Aufgabe | Modell |
|-------|-------|
| Vorhersage der Schlafqualität | Random Forest Regressor |
| Klassifikation von High Stress | Random Forest Classifier |

Die Modelle wurden anhand verschiedener Kennzahlen bewertet:

Regression:
- Root Mean Squared Error (RMSE)
- Mean Absolute Error (MAE)
- R² (Bestimmtheitsmaß)

Klassifikation:
- Accuracy
- Precision
- Recall
- F1-Score
- ROC-AUC

### 2.1 Vergleich der Modellperformance
Die folgende Übersicht zeigt die finalen Modellleistungen.

In [2]:
display(model_compare.style.hide(axis="index"))
display(clf_compare.style.hide(axis="index"))

Model,Root Mean Squared Error,Mean Absolute Error,Coefficient of Determination (R²)
Linear Regression,1.071506,0.850312,0.427285
Random Forest,1.101331,0.869489,0.394958


Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
Logistic Regression,0.990000,1.000000,0.936508,0.967213,0.999906
Random Forest,1.000000,1.000000,1.000000,1.000000,1.000000


In [3]:
#Recap aus Notebook 2
# --- Regressionsergebnisse vorbereiten
reg_compare = model_compare.copy()
reg_compare["Task"] = "Regression"
reg_compare = reg_compare.rename(columns={
    "Root Mean Squared Error": "RMSE",
    "Mean Absolute Error": "MAE",
    "Coefficient of Determination (R²)": "R²"
})

# --- Klassifikationsergebnisse vorbereiten
clf_overview = clf_compare.copy()
clf_overview["Task"] = "Classification"
clf_overview = clf_overview.rename(columns={
    "Accuracy": "Accuracy",
    "Precision": "Precision",
    "Recall": "Recall",
    "F1 Score": "F1",
    "ROC-AUC": "ROC-AUC"
})

# --- Long-Format
reg_long = reg_compare.melt(
    id_vars=["Model", "Task"],
    var_name="Metric",
    value_name="Value"
)

clf_long = clf_overview.melt(
    id_vars=["Model", "Task"],
    var_name="Metric",
    value_name="Value"
)

perf_long = pd.concat([reg_long, clf_long], ignore_index=True)

# --- Visualisierung
fig = px.bar(
    perf_long,
    x="Metric",
    y="Value",
    color="Model",
    facet_row="Task",
    barmode="group",
    template="simple_white",
    title="Model Performance Overview"
)

fig.update_layout(
    title_x=0.5,
    width=1100,
    height=700,
    legend_title_text=""
)

fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))

fig.show()

**Interpretation**

Der Vergleich zeigt Unterschiede zwischen den Modelltypen.

**Regression**

- Das Random-Forest-Modell erreicht eine bessere Vorhersagegüte als die lineare Regression.
- Dies deutet darauf hin, dass der Zusammenhang zwischen den Lifestyle-Variablen und der Schlafqualität teilweise **nichtlinear** ist.

**Klassifikation**

- Auch für die Klassifikation von High-Stress-Fällen erzielt der Random Forest höhere Werte bei F1-Score und ROC-AUC als die logistische Regression.
- Das Modell kann damit Stressfälle zuverlässiger identifizieren.

In beiden Aufgabenstellungen zeigt sich somit eine bessere Performance der **ensemblebasierten Random-Forest-Modelle**.

### 2.2 Stabilität der Modelle (Cross Validation)

Die Cross-Validation zeigt, wie stabil die Modellleistung über verschiedene Trainings-/Validierungsaufteilungen hinweg bleibt.

Zur Beurteilung der Stabilität werden die Fold-Ergebnisse als Boxplot dargestellt. So werden Streuung, Median und mögliche Ausreißer unmittelbar sichtbar.

In [4]:
#cv_plot = pd.DataFrame({
#    "Score": cast(list[float], cv_scores_lr) + cast(list[float], cv_scores_clf),
#    "Model": ["Linear Regression"] * len(cv_scores_lr) + # type: ignore
#             ["Random Forest"] * len(cv_scores_clf), # type: ignore
#    "Metric": ["R² (Regression)"] * len(cv_scores_lr) + # type: ignore
#              ["ROC-AUC (Classification)"] * len(cv_scores_clf) # type: ignore
#})

cv_plot = pd.DataFrame({
    "Score": [*cv_scores_lr, *cv_scores_clf],
    "Model": ["Linear Regression"] * len(cv_scores_lr) + ["Random Forest"] * len(cv_scores_clf),
    "Metric": ["R² (Regression)"] * len(cv_scores_lr) + ["ROC-AUC (Classification)"] * len(cv_scores_clf),
})

fig = px.box(
    cv_plot,
    x="Model",
    y="Score",
    color="Model",
    facet_col="Metric",
    points="all",
    template="simple_white",
    title="Cross-Validation – Modellstabilität"
)

fig.update_layout(
    title_x=0.5,
    width=950,
    height=420,
    legend_title_text=""
)

fig.for_each_annotation(
    lambda a: a.update(text=a.text.split("=")[-1])
)

fig.show()

**Interpretation**

Die Cross-Validation zeigt, wie stabil das Modell auf verschiedenen Trainings-/Validierungs-Splits performt.

Eine geringe Streuung der R²-Werte deutet darauf hin, dass das Modell robust gegenüber unterschiedlichen Trainingsdaten ist.

Starke Schwankungen würden dagegen darauf hindeuten, dass das Modell sehr stark von der jeweiligen Stichprobe abhängt.

### 2.3 Overfitting-Analyse

Die Overfitting-Analyse prüft, ob ein Modell auf den Trainingsdaten deutlich besser performt als auf unabhängigen Testdaten.

Ein starkes Leistungsgefälle zwischen Training und Test wäre ein Hinweis darauf, dass das Modell zu stark an die Trainingsdaten angepasst wurde und schlechter generalisiert.

Zur Beurteilung der Generalisierungsfähigkeit werden drei Perspektiven kombiniert:

- **Train Performance**
- **Cross-Validation Performance**
- **Test Performance**

Ein Modell gilt als robust, wenn diese drei Werte nahe beieinander liegen.
Starke Abweichungen zwischen Trainings- und Testleistung wären dagegen ein Hinweis auf Overfitting.

In [5]:
overfit_plot = pd.DataFrame({
    "Stage": ["Train","Test","Train","Test"],
    "Score": [
        perf.loc[perf.metric=="train_r2_rf","value"].values[0],
        perf.loc[perf.metric=="test_r2_rf","value"].values[0],
        perf.loc[perf.metric=="train_auc_rf","value"].values[0],
        perf.loc[perf.metric=="test_auc_rf","value"].values[0]
    ],
    "Task": ["Regression","Regression","Classification","Classification"]
})

fig = px.bar(
    overfit_plot,
    x="Stage",
    y="Score",
    color="Stage",
    facet_col="Task",
    template="simple_white",
    title="Overfitting Analyse – Train vs Test Performance"
)

fig.update_layout(
    title_x=0.5,
    width=950,
    height=420,
    legend_title_text=""
)

fig.for_each_annotation(
    lambda a: a.update(text=a.text.split("=")[-1])
)

fig.show()

In [6]:
# Werte aus perf holen
train_r2_rf = perf.loc[perf["metric"] == "train_r2_rf", "value"].values[0]
test_r2_rf  = perf.loc[perf["metric"] == "test_r2_rf", "value"].values[0]

train_auc_rf = perf.loc[perf["metric"] == "train_auc_rf", "value"].values[0]
test_auc_rf  = perf.loc[perf["metric"] == "test_auc_rf", "value"].values[0]

# Falls für Regression Linear Regression das finale Modell ist,
# nutze hier ihre CV-Werte und Test-Performance:
cv_reg_mean = cv_scores_lr.mean()
cv_reg_std  = cv_scores_lr.std(ddof=1)

# Test-R² der linearen Regression aus model_compare holen
test_r2_lr = model_compare.loc[
    model_compare["Model"] == "Linear Regression",
    "Coefficient of Determination (R²)"
].values[0]

# Für Train-R² der linearen Regression musst du diesen Wert aus Notebook 2 exportiert haben.
# Falls noch nicht vorhanden, vorerst näherungsweise leer lassen oder nachtragen.
train_r2_lr = perf.loc[perf["metric"] == "train_r2_lr", "value"].values[0]

# Klassifikation: Random Forest
cv_clf_mean = cv_scores_clf.mean()
cv_clf_std  = cv_scores_clf.std(ddof=1)

overfit_df = pd.DataFrame({
    "Task": ["Regression", "Regression", "Regression",
             "Classification", "Classification", "Classification"],
    "Stage": ["Train", "Cross-Validation", "Test",
              "Train", "Cross-Validation", "Test"],
    "Score": [train_r2_lr, cv_reg_mean, test_r2_lr,
              train_auc_rf, cv_clf_mean, test_auc_rf],
    "Error": [0, cv_reg_std, 0,
              0, cv_clf_std, 0],
    "Model": ["Linear Regression"] * 3 + ["Random Forest"] * 3
})

fig = px.bar(
    overfit_df,
    x="Stage",
    y="Score",
    color="Stage",
    error_y="Error",
    facet_col="Task",
    template="simple_white",
    title="Train vs Cross-Validation vs Test Performance"
)

fig.update_layout(
    title_x=0.5,
    width=950,
    height=450,
    legend_title_text=""
)

fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))

fig.show()

**Interpretation**

Die Grafik vergleicht die Modellleistung in drei Stufen:

- **Train**: Leistung auf den Trainingsdaten
- **Cross-Validation**: mittlere Leistung über mehrere Folds
- **Test**: Leistung auf unabhängigen Testdaten

**Regression vs. Klassifikation**:
- **Klassifikation** (rechts): Das Modell erreicht nahezu perfekte Scores (~1.0) in allen Phasen. Die Balken sind fast identisch hoch, was auf ein sehr leistungsstarkes und stabiles Modell hindeutet.
- **Regression** (links): Die Performance liegt deutlich niedriger (Score ca. 0.4 - 0.5). Das ist in der Praxis oft normal, zeigt aber, dass das Regressionsmodell die Varianz der Daten weniger präzise abbilden kann als das Klassifikationsmodell seine Kategorien.

**Woran erkennt man Overfitting?**

Ein Modell gilt als potenziell überangepasst, wenn:
- die Trainingsleistung deutlich höher ist als die Testleistung
- die Cross-Validation deutlich unterhalb der Trainingsleistung liegt
- die Ergebnisse stark zwischen den Folds schwanken

Bei beiden Modellen (Regression und Klassifikation) liegen die Balken für Train, Cross-Validation und Test sehr nah beieinander.  
**Fazit**: Es liegt kein signifikantes Overfitting vor. Das Modell hat nicht nur auswendig gelernt, sondern kann das Wissen auf neue, unbekannte Daten übertragen.

Die Fehlerbalken zeigen an, wie stark die Leistung schwankt, wenn man unterschiedliche Teile der Daten zum Trainieren nutzt.
Hier ist nur eine geringe Schwankung zu erkennen. Das Modell ist stabil und nicht abhängig von einem spezifischen Datensatz.

Während das Klassifikationsmodell besser aussieht (höherer Score), ist das Regressionsmodell dennoch gesund, da es konsistente Ergebnisse über alle Phasen liefert. Ein niedriger Score bei der Regression bedeutet oft einfach, dass das Problem komplexer ist oder mehr Daten benötigt werden, nicht zwangsläufig, dass das Modell schlecht ist.

## 3. Einflussfaktoren (Feature Importance)

Ziel dieses Kapitels ist die Interpretation der wichtigsten Variablen, die in den trainierten Modellen zur Vorhersage beitragen.

Es werden zwei Modelltypen betrachtet:

• Regressionsmodell (Sleep Quality)  
• Klassifikationsmodell (High Stress)

Die Analyse basiert auf der Feature Importance der Random-Forest-Modelle, die im Modelltraining berechnet wurden.

Die Werte geben an, wie stark ein Feature zur Modellentscheidung beiträgt. Sie sind **keine kausalen Effekte**, sondern modellbasierte Relevanzindikatoren.

3.1 Import:

### 3.1 Wichtigste Variablen im Modellvergleich

In [7]:
display(
    feat_reg
    .sort_values("importance", ascending=False)
    .style
    .format({"importance": "{:.3f}"})
    .hide(axis="index")
)

feature,importance
Stress Level (1–10),0.549
Sugar Intake (g/day),0.087
Physical Activity (min/day),0.082
Age,0.075
Sleep Duration (hours),0.074
Alcohol (drinks/week),0.073
Heart Rate (bpm),0.060


**Interpretation der Feature Importance**

Die Feature Importance des Random-Forest-Modells zeigt eine **stark asymmetrische Einflussstruktur**.

**Stress** besitzt mit einer Importance von über 0.5 den mit Abstand größten Beitrag zur Vorhersage der Schlafqualität.

Alle anderen Variablen tragen jeweils weniger als 10 % zur Modellentscheidung bei.

Dies deutet darauf hin, dass Schlafqualität im Datensatz primär durch das wahrgenommene Stressniveau bestimmt wird.

Lifestyle-Faktoren wie Zuckeraufnahme oder körperliche Aktivität spielen eine deutlich geringere Rolle.

In [8]:
feat_sorted = feat_reg.sort_values(
    "importance",
    ascending=False
).copy()

feat_sorted["cumulative_importance"] = feat_sorted["importance"].cumsum()

fig = go.Figure()

# Balken
fig.add_bar(
    x=feat_sorted["feature"],
    y=feat_sorted["importance"],
    name="Importance",
    width=0.42
)

# kumulative Linie
fig.add_scatter(
    x=feat_sorted["feature"],
    y=feat_sorted["cumulative_importance"],
    mode="lines+markers",
    name="Cumulative Importance",
    yaxis="y2",
    line=dict(width=3),
    marker=dict(size=7)
)

# 80%-Referenzlinie
fig.add_scatter(
    x=[feat_sorted["feature"].iloc[0], feat_sorted["feature"].iloc[-1]],
    y=[0.8, 0.8],
    mode="lines",
    name="80% Threshold",
    yaxis="y2",
    line=dict(dash="dot", color="gray")
)

fig.update_layout(
    template="simple_white",
    title="Pareto Chart – Feature Importance",
    title_x=0.45,
    width=1050,
    height=480,
    bargap=0.06,
    xaxis=dict(
        title="Feature",
        tickangle=-20
    ),
    yaxis=dict(
        title="Feature Importance"
    ),
    yaxis2=dict(
        title="Cumulative Importance",
        overlaying="y",
        side="right",
        range=[0, 1.05]
    ),
    legend=dict(
        orientation="h",
        x=0.5,
        y=0.95,
        xanchor="center",
        yanchor="bottom"
    ),
    margin=dict(l=70, r=190, t=80, b=90)
)

fig.show()

### 3.2 Struktur der Einflussverteilung

Der Pareto-Chart zeigt, dass sich die Modellinformation stark auf wenige Variablen konzentriert.

Bereits das wichtigste Feature (Stress Level) erklärt mehr als die Hälfte der modellrelevanten Information. Die übrigen Variablen tragen jeweils nur einen vergleichsweise kleinen Anteil zur Vorhersage bei.

Diese Struktur weist darauf hin, dass die Vorhersage der Schlafqualität im Datensatz primär durch eine dominante Belastungsdimension bestimmt wird, die stark mit dem Stressniveau zusammenhängt.

### 3.3 Modellkomplexität und Dominanz einzelner Variablen

Die kumulative Feature Importance zeigt, dass bereits wenige Variablen den Großteil der Modellinformation erklären.

Im vorliegenden Modell wird die Schwelle von 80 % der kumulativen Importance bereits nach wenigen Features erreicht. Dies deutet darauf hin, dass die relevante Struktur der Daten durch eine kleine Anzahl zentraler Variablen beschrieben werden kann.

Das Modell benötigt daher nur eine begrenzte Anzahl von Einflussfaktoren, um die Schlafqualität zuverlässig vorherzusagen.

**Wie ungleich verteilt ist die Modellinformation über die Features?**

In [9]:
feat_sorted = feat_reg.sort_values("importance", ascending=False).copy()
feat_sorted["cum_importance"] = feat_sorted["importance"].cumsum()

n = len(feat_sorted)


feat_sorted["feature_share"] = np.arange(1, n + 1) / n


equal_x = np.arange(0, n + 1) / n
equal_y = np.arange(0, n + 1) / n


curve_x = np.concatenate([[0], feat_sorted["feature_share"].to_numpy()])
curve_y = np.concatenate([[0], feat_sorted["cum_importance"].to_numpy()])


fig = go.Figure()


fig.add_trace(
    go.Scatter(
        x=equal_x,
        y=equal_y,
        mode="lines",
        name="Equal distribution",
        line=dict(dash="dash", color="gray")
    )
)


fig.add_trace(
    go.Scatter(
        x=curve_x,
        y=curve_y,
        mode="lines+markers",
        name="Observed concentration",
        line=dict(width=3)
    )
)

fig.update_layout(
    template="simple_white",
    title="Information Concentration Curve",
    title_x=0.5,
    width=850,
    height=500,
    xaxis_title="Share of Features",
    yaxis_title="Cumulative Share of Model Information",
    legend=dict(
        orientation="h",
        x=0.5,
        y=1.05,
        xanchor="center"
    )
)

fig.show()

top1 = feat_sorted["importance"].iloc[0]
top3 = feat_sorted["importance"].iloc[:3].sum()

print(f"Top 1 feature explains {top1:.1%} of model information.")
print(f"Top 3 features explain {top3:.1%} of model information.")

Top 1 feature explains 54.9% of model information.
Top 3 features explain 71.8% of model information.


Die Information Concentration Curve zeigt, wie ungleich die Modellinformation über die Features verteilt ist.

- Die gestrichelte Diagonale entspricht einer hypothetischen Gleichverteilung:
  Jedes Feature würde denselben Beitrag leisten.
- Die beobachtete Kurve liegt deutlich oberhalb dieser Linie.

**Interpretation**

Die beiden obigen Darstellungen liefern komplementäre Perspektiven auf die Struktur der Modellinformation.

Der Pareto-Chart zeigt die Rangordnung der Variablen nach ihrer modellbasierten Bedeutung.  
Dabei wird deutlich, dass das Stressniveau mit Abstand den größten Beitrag zur Vorhersage der Schlafqualität leistet. Bereits das wichtigste Feature erklärt mehr als die Hälfte der gesamten modellrelevanten Information, während alle übrigen Variablen jeweils nur vergleichsweise kleine Beiträge liefern.

Die Information Concentration Curve ergänzt diese Darstellung, indem sie die Verteilung der Modellinformation über alle Features hinweg betrachtet.  
Die beobachtete Kurve liegt deutlich oberhalb der Diagonalen einer hypothetischen Gleichverteilung. Dies zeigt, dass die Modellinformation stark auf wenige Variablen konzentriert ist.

Zusammen genommen weisen beide Darstellungen darauf hin, dass die Vorhersage der Schlafqualität im Datensatz durch eine stark dominante Einflussstruktur geprägt ist. Ein kleiner Teil der Variablen trägt den überwiegenden Anteil der Modellinformation, während viele weitere Variablen nur einen begrenzten zusätzlichen Beitrag leisten.

Diese Struktur deutet darauf hin, dass die zugrunde liegende Datenstruktur nicht durch viele gleichgewichtige Einflussfaktoren bestimmt wird, sondern durch eine zentrale Belastungsdimension, die insbesondere mit dem Stressniveau zusammenhängt.

## 4 Hypothesenprüfung

Im Folgenden werden die in Notebook 1 formulierten Hypothesen anhand der Ergebnisse aus
EDA, Feature Engineering, Clustering und Modellierung überprüft.

<div style="
  border-left: 6px solid #2e7d32;
  background-color: #f1faf3;
  padding: 14px 18px;
  margin: 12px 0;
  font-size: 16px;
  line-height: 1.5;
  border-radius: 6px;
">
  <strong>Wichtig:</strong><br>
  Die Bewertung erfolgt nicht kausal, sondern datenbasiert und modellgestützt.
  Die Formulierungen "bestätigt" bzw. "nicht bestätigt" bedeuten daher:<br>
  durch die vorliegenden Analysen gestützt oder nicht gestützt.
</div>

### 4.1 Hypothesen zur Schlafqualität (Regression)

#### H1: Stress ist der stärkste negative Prädiktor der Schlafqualität.

**Bewertung:** **weitgehend bestätigt**

**Begründung:**
- Bereits in der explorativen Analyse zeigte sich ein starker negativer Zusammenhang zwischen Stress und Schlafqualität.
- In der PCA war Stress Teil der dominanten Belastungsdimension (PC1).
- In den Regressionsanalysen gehört Stress zu den zentralen Einflussfaktoren auf Schlafqualität.
- Inhaltlich ist der Zusammenhang konsistent: höherer Stress geht mit geringerer subjektiver Schlafqualität einher.

**Einschränkung:**
- Falls Schlafdauer oder ein abgeleitetes Merkmal ähnlich stark wirkt, wäre die Hypothese eher nur **teilweise bestätigt**.

**Fazit:**  
- Dass Stress tatsächlich der **stärkste** Prädiktor ist, wurde durch die UNtersuchung zu Feature-Importances belegt.


#### H2: Schlafdauer wirkt protektiv auf die Schlafqualität.

**Bewertung:** **bestätigt**

**Begründung:**
- Die explorativen Analysen zeigten einen positiven Zusammenhang zwischen Schlafdauer und Schlafqualität.
- In den Scatter- und Trendanalysen war längerer Schlaf mit besserer Schlafqualität verbunden.
- Schlafdauer war sowohl in der EDA als auch in der Modellierung ein relevanter Prädiktor.
- Die Hypothese ist auch inhaltlich plausibel und konsistent mit dem Belastungsprofil aus PC1.

**Fazit:**  
Die Hypothese ist durch EDA und Modellierung gut gestützt.


#### H3: Alkohol beeinflusst Schlafqualität nichtlinear.

**Bewertung:** **teilweise bestätigt / vorsichtig formulieren**

**Begründung:**
- In Notebook 1 wurde bereits ein möglicher Schwellen- bzw. Krümmungseffekt diskutiert.
- LOWESS-Analysen und die Schwellenmarkierung im oberen Bereich des Alkoholkonsums deuteten auf einen nichtlinearen Verlauf hin.
- Die Beobachtung war jedoch eher explorativ als modellbasiert abgesichert.

**Gegenargument / Einschränkung:**
- In den späteren Regressionsmodellen war Alkohol nach bisherigem Verlauf nicht der dominante Einflussfaktor.

**Fazit:**  
Die Hypothese ist **explorativ plausibel**, aber **nicht stark modellgestützt bestätigt**. Daher besser:  
"Es gibt Hinweise auf einen nichtlinearen Einfluss von Alkohol auf Schlafqualität."


### 4.2 Hypothesen zu Stress (Klassifikation)

#### H4: Stress ist entlang einer latenten Belastungsdimension linear klassifizierbar.

**Bewertung:** **teilweise bestätigt**

**Begründung:**
- Bereits in der EDA zeigte sich eine starke Struktur entlang PC1.
- Die PCA deutete auf eine Belastungsachse hin, auf der sich stress-nahe Muster bündeln.
- Die linearen Modelle erzielten bereits eine brauchbare Klassifikationsleistung.
- Damit ist die Grundidee einer linearen Separierbarkeit nicht unbegründet.

**Einschränkung:**
- In der Modellierung war der Random Forest für die Klassifikation stärker als das lineare Modell.
- Das spricht dafür, dass die Struktur zwar **teilweise linear**, aber nicht vollständig linear ist.
- Die Hypothese wäre vollständig bestätigt, wenn das lineare Modell bereits nahezu die finale Leistung erreicht hätte.

**Fazit:**  
Die Daten sprechen für eine **erkennbare lineare Belastungsdimension**, aber nicht für eine vollständig lineare Klassifizierbarkeit.  


#### H5: Herzfrequenz ist ein zentraler physiologischer Marker für hohen Stress.

**Bewertung:** **eher bestätigt**

**Begründung:**
- Bereits in Notebook 1 zeigte sich ein Zusammenhang zwischen Stress und Herzfrequenz.
- Herzfrequenz war in den Moderations- und Belastungsanalysen relevant.
- In den Klassifikationsmodellen ist Herzfrequenz inhaltlich ein plausibler und physiologisch sinnvoller Marker.
- Auch in der PCA lag Herzfrequenz auf derselben Belastungsstruktur wie Stress.

**Einschränkung:**
- Ob Herzfrequenz ein **zentraler** Marker ist, muss über die finale Feature-Importance bzw.
über die Rolle im Klassifikationsmodell konkret belegt werden.
- Falls Schlafdauer oder Schlafqualität im Modell stärker gewichtet würden, wäre Herzfrequenz eher ein **wichtiger**, aber nicht der wichtigste Marker.

**Fazit:**  
Die Hypothese ist durch die bisherigen Analysen gut gestützt, könnte aber mitels Feature Importance noch präzise begründet werden.


### 4.3 Cluster-Hypothesen

#### H6: Es existieren keine diskreten Lifestyle-Cluster.

**Bewertung:** **weitgehend bestätigt**

**Begründung:**
- Bereits die PCA zeigte eher kontinuierliche Übergänge als scharf getrennte Gruppen.
- Die Cluster lagen im PCA-Raum nicht als klar separierte Inseln vor, sondern mit Überlappungen.
- Auch die Diskussion zu 6.3 und 6.4 deutete eher auf kontinuierliche Ordnung als auf harte Segmentierung.
- Diese Beobachtung war auch die Grundlage der Cluster-Hypothese.

**Wichtige Einschränkung:**
- K-Means erzeugt technisch immer Cluster, auch wenn die Daten kontinuierlich sind.
- Die Existenz algorithmischer Cluster ist daher **kein Beweis** für diskrete Lifestyle-Typen.

**Fazit:**  
Die Daten sprechen eher für eine **kontinuierliche Struktur mit graduellen Profilen**  
als für scharf abgegrenzte Lifestyle-Klassen.  
Die Hypothese ist damit gut gestützt.


#### H7: Health Risk ist Projektion einer Belastungsdimension und kein eigenständiger Cluster.

**Bewertung:** **klar bestätigt**

**Begründung:**
- In der PCA zeigte sich eine monotone Anordnung der Risk-Gruppen entlang PC1.
- Die Mittelwerte der Gruppen waren nicht zufällig verteilt, sondern systematisch verschoben.
- Die beobachtete Effektstärke (Cohen’s d) war sehr hoch.
- Die Gruppen lagen nicht als eigene Clusterinseln vor, sondern wie Projektionen entlang einer gemeinsamen Belastungsachse.

**Bedeutung:**
- Health Risk verhält sich damit eher wie eine **ordinale Projektion** einer latenten Gesundheits-/Belastungsdimension.
- Die Risiko-Klassen repräsentieren demnach weniger "verschiedene Typen" als vielmehr verschiedene Positionen auf derselben Achse.

**Fazit:**  
Diese Hypothese ist durch PCA, Mittelwertverschiebung und Effektstärke besonders stark gestützt.


### 4.4 Zusammenfassende Bewertung der Hypothesen

| Hypothese | Bewertung | Kurzbegründung |
|---|---|---|
| **H1** Stress ist stärkster negativer Prädiktor der Schlafqualität | **weitgehend bestätigt** | starker negativer Zusammenhang, aber Rangordnung final noch explizit prüfen |
| **H2** Schlafdauer wirkt protektiv auf Schlafqualität | **bestätigt** | konsistent positiver Zusammenhang in EDA und Modellierung |
| **H3** Alkohol wirkt nichtlinear auf Schlafqualität | **teilweise bestätigt** | explorative LOWESS-Hinweise, aber noch keine starke Modellabsicherung |
| **H4** Stress ist entlang einer Belastungsdimension linear klassifizierbar | **teilweise bestätigt** | lineare Struktur vorhanden, aber RF deutet auf zusätzliche Nichtlinearität hin |
| **H5** Herzfrequenz ist zentraler physiologischer Marker für hohen Stress | **eher bestätigt** | inhaltlich und empirisch relevant |
| **H6** Es existieren keine diskreten Lifestyle-Cluster | **weitgehend bestätigt** | kontinuierliche statt scharf getrennter Struktur |
| **H7** Health Risk ist Projektion einer Belastungsdimension | **klar bestätigt** | monotone Verschiebung entlang PC1, sehr hohe Effektstärken |

### 4.5 Gesamtfazit

Die Hypothesenprüfung zeigt insgesamt ein konsistentes Bild:

- Schlafqualität wird vor allem durch **Belastung und Schlafverhalten** beeinflusst.
- Hohes Stressniveau ist teilweise linear erklärbar, enthält aber auch **nichtlineare Strukturanteile**.
- Die Daten sprechen eher für eine **kontinuierliche Belastungsstruktur** als für scharf getrennte Lifestyle-Typen.
- Besonders stark bestätigt wird die Annahme, dass **Health Risk keine eigenständige Clusterstruktur**, sondern eine Projektion einer übergeordneten Belastungsdimension ist.

## 5 Segmentstruktur & Lifestyle-Typen
In diesem Abschnitt werden die bereits in Notebook 2 identifizierten Cluster **nicht erneut berechnet**, sondern inhaltlich interpretiert. 

Zur kompakten Interpretation werden drei zentrale Kennzahlen pro Cluster gegenübergestellt:
- durchschnittliche Schlafqualität
- Anteil hoher Stressfälle
- durchschnittlicher Risk Score


In [10]:
# Segmentstruktur der Cluster
fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=(
        "Mean Sleep Quality",
        "Share High Stress",
        "Mean Risk Score"
    ),
    horizontal_spacing=0.05
)

fig.add_trace(
    go.Bar(
        x=cluster_summary["cluster_label"],
        y=cluster_summary["sleepQuality"],
        name="Mean Sleep Quality",
        width=0.45,
        showlegend=False
    ),
    row=1,
    col=1
)

fig.add_trace(
    go.Bar(
        x=cluster_summary["cluster_label"],
        y=cluster_summary["stressHigh"],
        name="Share High Stress",
        width=0.45,
        showlegend=False
    ),
    row=1,
    col=2
)

fig.add_trace(
    go.Bar(
        x=cluster_summary["cluster_label"],
        y=cluster_summary["riskScore"],
        name="Mean Risk Score",
        width=0.45,
        showlegend=False
    ),
    row=1,
    col=3
)

fig.update_layout(
    template="simple_white",
    title="Segmentstruktur der Cluster",
    title_x=0.5,
    width=1000,
    height=420,
    bargap=0.35
)

fig.update_yaxes(title_text="Sleep Quality", row=1, col=1)
fig.update_yaxes(title_text="Share", row=1, col=2)
fig.update_yaxes(title_text="Risk Score", row=1, col=3)

fig.show()

**Interpretation**

Die drei Panels zeigen die Cluster entlang zentraler Ergebnisgrößen:

- **Mean Sleep Quality**: durchschnittliche Schlafqualität pro Cluster
- **Share High Stress**: Anteil hoher Stressfälle pro Cluster
- **Mean Risk Score**: mittlerer Risk Score pro Cluster

Die Darstellung macht sichtbar, dass sich die Cluster entlang einer gemeinsamen Belastungsdimension staffeln:

- Cluster mit höherem Stressanteil weisen tendenziell geringere Schlafqualität auf
- Cluster mit höherem Risk Score liegen ebenfalls eher im belasteten Bereich

Die Unterschiede erscheinen dabei eher graduell als strikt getrennt, was die Interpretation der Cluster als kontinuierliche Lifestyle-Profile unterstützt.

**Einordnung in die Hypothesenprüfung**

Die Segmentanalyse stützt insbesondere zwei Hypothesen:

- **H6:** Es existieren keine diskreten Lifestyle-Cluster.
- **H7:** Health Risk ist Projektion einer Belastungsdimension und kein eigenständiger Cluster.

Die Cluster verhalten sich somit eher wie ordinal geordnete Profile als wie klar voneinander getrennte Gruppen.

## 6. Kritische Reflexion

Die vorliegende Analyse liefert strukturierte Einblicke in die Zusammenhänge zwischen Stress, Schlafqualität und Lifestyle-Faktoren. Gleichzeitig weist die Untersuchung mehrere methodische und datenbezogene Einschränkungen auf, die bei der Interpretation der Ergebnisse berücksichtigt werden müssen.

#### Datengrundlage und Datengröße

Der verwendete Datensatz umfasst rund 2.000 Beobachtungen und erlaubt damit eine stabile explorative Analyse sowie das Training einfacher Machine-Learning-Modelle. Für komplexere Modellstrukturen oder eine robuste Generalisierung auf unterschiedliche Populationen wäre jedoch eine größere und heterogenere Datengrundlage wünschenswert.

Zudem weist die Datenstruktur darauf hin, dass der Datensatz synthetisch generiert wurde. Dies zeigt sich unter anderem daran, dass einige in realen Gesundheitsdaten erwartbare Zusammenhänge fehlen oder stark vereinfacht erscheinen.

Ein Beispiel hierfür ist der Zusammenhang zwischen BMI und Gesundheitsrisiko. In realen epidemiologischen Datensätzen wäre zumindest ein moderater Zusammenhang zwischen BMI und gesundheitlichen Risiken zu erwarten. Das Ausbleiben eines solchen Effekts spricht dafür, dass:

- der Datensatz synthetisch erzeugt wurde,
- die Risikologik auf andere Variablengruppen fokussiert ist,
- oder einzelne Variablen bewusst voneinander entkoppelt modelliert wurden.

Die Ergebnisse spiegeln daher eher die Struktur des simulierten Systems als reale medizinische Zusammenhänge wider.

#### Modellannahmen und methodische Grenzen

Die eingesetzten Modelle (lineare Regression, Random Forest, logistische Regression) basieren auf bestimmten Annahmen über die Datenstruktur. Obwohl Random-Forest-Modelle relativ flexibel sind, bleibt ihre Interpretation modellabhängig.

Die Feature Importance liefert Hinweise darauf, welche Variablen für die Modellentscheidung relevant sind. Sie erlaubt jedoch keine direkte Aussage über kausale Wirkzusammenhänge oder über die Richtung einzelner Effekte.

Darüber hinaus zeigte die Analyse eine starke Konzentration der Modellinformation auf wenige Variablen, insbesondere auf das Stressniveau. Diese Dominanz kann mehrere Ursachen haben:

- eine tatsächliche strukturelle Bedeutung dieser Variable,
- eine vereinfachte Datenlogik im synthetischen Datensatz,
- oder eine hohe Korrelation mit weiteren nicht beobachteten Faktoren.

Die Feature Importance ist daher als **modellbasierte Relevanzstruktur** zu verstehen und nicht als kausale Erklärung.

#### Korrelation versus Kausalität

Ein zentraler methodischer Aspekt betrifft die Unterscheidung zwischen Korrelation und Kausalität. Die explorative Analyse und die trainierten Modelle identifizieren statistische Zusammenhänge zwischen Variablen, erlauben jedoch keine Aussagen über Ursache-Wirkungs-Beziehungen.

Beispielsweise kann ein Zusammenhang zwischen Stressniveau und Schlafqualität sowohl darauf hinweisen, dass Stress die Schlafqualität reduziert, als auch darauf, dass schlechter Schlaf das Stressniveau erhöht. Ohne longitudinale Daten oder experimentelle Designs lassen sich solche Wirkungsrichtungen nicht eindeutig bestimmen.

Die Ergebnisse sollten daher als **deskriptive Struktur der Daten** interpretiert werden und nicht als kausale Gesundheitsmodelle.

#### Generalisierbarkeit der Ergebnisse

Da der Datensatz synthetisch generiert und in seinem Variablenumfang begrenzt ist, lassen sich die Ergebnisse nur eingeschränkt auf reale Populationen übertragen.

Insbesondere fehlen mehrere Faktoren, die in realen Gesundheitsstudien relevant wären, beispielsweise:

- sozioökonomische Variablen
- medizinische Vorerkrankungen
- psychologische Faktoren
- langfristige Lebensstilindikatoren

Die Modelle beschreiben daher primär die Struktur des vorliegenden Datensatzes und nicht notwendigerweise allgemeingültige Gesundheitszusammenhänge.

#### Grenzen der Analyse

Neben den datenbezogenen Einschränkungen ergeben sich auch analytische Grenzen aus der gewählten Methodik.

Die Analyse konzentriert sich auf eine Kombination aus explorativer Datenanalyse und relativ einfachen Machine-Learning-Modellen. Komplexere Modellierungsansätze (z. B. kausale Modelle, longitudinale Analysen oder tiefere Feature-Interaktionen) wurden bewusst nicht eingesetzt, um die Interpretierbarkeit der Ergebnisse zu erhalten.

Auch die Clusteranalyse zeigte keine klar separierten Lifestyle-Typen, sondern eher eine kontinuierliche Struktur der Daten. Dies deutet darauf hin, dass Gesundheitsverhalten im Datensatz eher entlang gradueller Dimensionen als in klar abgegrenzten Gruppen organisiert ist.

#### Zusammenfassung der Limitationen

Zusammenfassend ergeben sich mehrere zentrale Einschränkungen der Analyse:

- synthetische Datengenerierung und vereinfachte Datenlogik  
- begrenzte Variablenauswahl und fehlende medizinische Kontextfaktoren  
- keine kausale Interpretierbarkeit der Modelle  
- eingeschränkte Generalisierbarkeit der Ergebnisse  
- mögliche Dominanz einzelner Variablen durch die Struktur des Datensatzes  

Trotz dieser Einschränkungen ermöglicht die Analyse eine systematische Untersuchung der Datenstruktur und liefert nachvollziehbare Einblicke in die modellbasierte Bedeutung zentraler Gesundheitsvariablen.

## 7. Dashboard

### 7.1 Ziel des Dashboards

Ziel des Dashboards ist es, die im Projekt gewonnenen Erkenntnisse zur Beziehung zwischen **Schlafqualität, Stress und Lebensstilfaktoren** interaktiv zugänglich zu machen. Während die vorhergehenden Analysen in den Notebooks überwiegend statisch erfolgen, ermöglicht das Dashboard eine explorative Untersuchung der Daten sowie eine transparente Darstellung der Modellresultate.

Das Dashboard verfolgt dabei drei zentrale Ziele:

**Interaktive Exploration der Daten**

Nutzerinnen und Nutzer können den Datensatz über verschiedene Filter dynamisch untersuchen. Dadurch lassen sich Zusammenhänge zwischen Stress, Schlafdauer, Schlafqualität und weiteren Lebensstilfaktoren unmittelbar analysieren.

**Darstellung der Kernerkenntnisse der Analyse**

Die Ergebnisse der explorativen Datenanalyse sowie der Clusteranalyse werden visuell aufbereitet. Dadurch können typische Muster im Datensatz, etwa Gruppen mit erhöhtem Stressniveau oder erhöhtem Gesundheitsrisiko, schnell erkannt werden.

**Transparenz der Modellleistung**

Das Dashboard integriert Ergebnisse aus dem Modellierungsprozess, insbesondere die **Feature Importance des Random-Forest-Modells**, das zur Vorhersage der Schlafqualität eingesetzt wurde. Dadurch wird nachvollziehbar, welche Einflussfaktoren besonders relevant für die Modellentscheidung sind.

Durch diese Kombination aus Exploration, Ergebnisdarstellung und Modelltransparenz fungiert das Dashboard als **interaktive Zusammenfassung der gesamten Analysepipeline** des Projekts.

### 7.2 Struktur des Dashboards

Das Dashboard wurde mit **Plotly Dash** implementiert und folgt einer tab-basierten Struktur. Eine globale Sidebar ermöglicht die Filterung des Datensatzes, während die einzelnen Tabs unterschiedliche Aspekte der Analyse visualisieren.

Die Sidebar enthält folgende Filteroptionen:

- Alter (Range-Slider)
- Geschlecht
- Raucherstatus
- BMI-Kategorie
- Clusterzugehörigkeit

Alle Visualisierungen reagieren dynamisch auf diese Filter und aktualisieren sich automatisch über Dash-Callbacks.

#### Tab Explore – Interaktive Exploration

Der Tab `Explore` dient der explorativen Analyse der Beziehungen zwischen Stress, Schlaf und weiteren Lebensstilfaktoren.

Die Visualisierung umfasst:

- **Scatterplot:** Stressniveau vs. Schlafqualität
- **Boxplot:** Schlafdauer nach Clusterzugehörigkeit

Der Scatterplot ermöglicht es, Zusammenhänge zwischen Stress und Schlafqualität zu untersuchen. Durch die farbliche Kodierung der Cluster wird sichtbar, ob bestimmte Gruppen systematisch höhere oder niedrigere Schlafqualität aufweisen.

Die Boxplots zeigen zusätzlich die Verteilung der Schlafdauer innerhalb der identifizierten Cluster und unterstützen so die Interpretation der Clusterstruktur.

#### Tab Clusters

Der Tab `Clusters` visualisiert die Ergebnisse der in Notebook 2 durchgeführten **Clusteranalyse**, die auf Stress-, Schlaf- und Lifestyle-Variablen basiert.

Zwei Visualisierungen stehen im Mittelpunkt:

**Clustervergleich**

Hier können verschiedene Kennzahlen ausgewählt werden, beispielsweise:

- durchschnittliche Schlafqualität
- Anteil hoher Stresswerte
- durchschnittlicher Gesundheitsrisikoscore

Die Balkendiagramme zeigen die Unterschiede zwischen den Clustern und verdeutlichen typische Profile innerhalb der Population.

**Clustergrößen**

Ein weiteres Diagramm zeigt die relative Größe der einzelnen Cluster. Neben der absoluten Anzahl der Beobachtungen wird auch der prozentuale Anteil dargestellt.

Dies erleichtert die Einschätzung, welche Lebensstilprofile im Datensatz besonders häufig vorkommen.

#### Tab Sleep Quality

Der Tab `Sleep Quality`  stellt die Ergebnisse des **Random-Forest-Regressionsmodells** dar, das zur Vorhersage der Schlafqualität trainiert wurde.

Das Diagramm zeigt die **wichtigsten Einflussvariablen (Features)** des Modells. Dazu gehören unter anderem:

- Stressniveau
- Schlafdauer
- körperliche Aktivität
- Herzfrequenz

Über einen Slider kann die Anzahl der dargestellten Top-Features angepasst werden. Zusätzlich wird die kumulierte Importance der ausgewählten Variablen angezeigt.

Diese Darstellung ermöglicht eine transparente Interpretation des Modells und zeigt, welche Faktoren den größten Einfluss auf die Schlafqualität haben.

#### Tab 3D View

Der Tab `3D View`  erweitert die Analyse um eine dreidimensionale Visualisierung zentraler Variablen.

Dargestellt werden:

- Stressniveau
- Schlafdauer
- Schlafqualität

Die Punkte werden nach Clusterzugehörigkeit eingefärbt. Dadurch lassen sich räumliche Strukturen im Datensatz erkennen, beispielsweise Gruppen mit

- hohem Stress und niedriger Schlafqualität
- niedrigem Stress und hoher Schlafqualität.

Die interaktive 3D-Darstellung ermöglicht eine freie Rotation der Ansicht und erleichtert so das Erkennen komplexer Zusammenhänge zwischen den Variablen.

### 7.3 Schematische Architektur des Dashboards

Die technische Struktur des Dashboards folgt dem reaktiven Konzept von Plotly Dash, bei dem Eingabeelemente, Datenverarbeitung und Visualisierung über Callbacks miteinander verbunden sind.

Die grundlegende Architektur lässt sich wie folgt darstellen:

```mermaid
flowchart TD
    A["Datenquellen<br/>dashboard_data.csv<br/>cluster_summary.csv<br/>feature_importance_rf.csv"] --> B["Datenaufbereitung<br/>Pandas DataFrames"]
    B --> C["Filter-Interface<br/>Age Slider<br/>Gender Dropdown<br/>Smoking Dropdown<br/>BMI Dropdown<br/>Cluster Dropdown"]
    C --> D["Dash Callbacks<br/>Filterung<br/>KPI-Berechnung<br/>Aktualisierung der Visualisierungen"]
    D --> E["Explore Tab<br/>Scatterplot<br/>Boxplot"]
    D --> F["Clusters Tab<br/>Clustervergleich<br/>Clustergrößen"]
    D --> G["Sleep Quality Tab<br/>Top-N Features<br/>Kumulierte Importance"]
    D --> H["3D View Tab<br/>3D-Scatterplot"]

In [11]:
# Dashboard öffnen
from IPython.display import HTML

html_code = '''
<div style="padding: 10px; background-color: #f0f7ff; border-radius: 8px; border-left: 4px solid #0066cc;">
    <a href="http://localhost:8050" target="_blank"
       style="background-color: #0066cc; color: white; padding: 10px 20px; 
              text-decoration: none; border-radius: 5px; display: inline-block; 
              font-weight: bold;">
        Dashboard im Browser öffnen
    </a>
    <p style="color: #666; margin-top: 10px; font-size: 0.9em;">
        Alternativ: <a href="http://localhost:8050">https://localhost:8050</a>
    </p>
</div>
'''

HTML(html_code)

## 8. Abschluss

#### Einordnung des Projekts

Die vorliegende Arbeit verfolgt primär einen **explorativen Data-Science-Ansatz**.  
Ziel war nicht die Ableitung kausaler wissenschaftlicher Aussagen, sondern die systematische Untersuchung eines Gesundheitsdatensatzes mithilfe moderner Methoden der **Explorativen Datenanalyse (EDA)** und **prädiktiver Modellierung**.

Der Schwerpunkt lag auf folgenden Aspekten:

• strukturelles Verständnis des Datensatzes  
• Identifikation statistischer Muster und Zusammenhänge  
• Entwicklung einfacher Vorhersagemodelle  
• Interpretation modellbasierter Einflussfaktoren  

Die Ergebnisse sind daher als **datengetriebene Strukturinterpretation** zu verstehen und nicht als medizinisch-kausale Schlussfolgerungen.


#### Zentrale Erkenntnisse der Analyse

Die explorative Analyse zeigte eine klare Struktur innerhalb des Datensatzes.

Eine dominante Belastungsdimension verband insbesondere:

- Stressniveau  
- Schlafqualität  
- Schlafdauer  
- Herzfrequenz

Diese Struktur wurde sowohl in der **Korrelationsanalyse**, der **PCA**, als auch in den **Modellergebnissen** sichtbar.

Die Regressionsanalyse zeigte, dass insbesondere das Stressniveau einen starken Einfluss auf die Vorhersage der Schlafqualität besitzt. Die Feature-Importance-Analyse bestätigte eine **stark konzentrierte Einflussstruktur**, in der wenige Variablen den größten Teil der Modellinformation tragen.

Die Klassifikationsmodelle zur Vorhersage eines hohen Stressniveaus erreichten ebenfalls stabile Ergebnisse und zeigen, dass physiologische und schlafbezogene Variablen als Indikatoren für erhöhte Belastung dienen können.

Die Clusteranalyse ergab dagegen **keine klar abgegrenzten Lifestyle-Typen**, sondern eher eine kontinuierliche Struktur entlang einer Belastungsdimension.


#### Methodische Einordnung

Das Projekt zeigt exemplarisch, wie ein Data-Science-Workflow von der Datenexploration bis zur Modellinterpretation aufgebaut werden kann.

Die Analyse folgte dabei einer typischen Pipeline:

1. Datenverständnis und Exploration  
2. Feature Engineering  
3. Strukturuntersuchung (Korrelation, PCA, Clustering)  
4. Modellierung (Regression und Klassifikation)  
5. Evaluation und Interpretation der Modelle  
6. kritische Reflexion der Ergebnisse  

Der Schwerpunkt lag dabei bewusst auf **Interpretierbarkeit und Nachvollziehbarkeit**, nicht auf maximaler Modellkomplexität.


#### Bedeutung der Ergebnisse

Die Ergebnisse zeigen, dass bereits mit relativ einfachen Methoden relevante Strukturen in Daten sichtbar gemacht werden können.

Insbesondere die Kombination aus:

- Explorative Analyse  
- Modellierung  
- Feature-Interpretation  

ermöglicht ein konsistentes Bild der zugrundeliegenden Datenstruktur.

Das Projekt demonstriert damit einen typischen **EDA-getriebenen Analyseprozess**, wie er in vielen praktischen Data-Science-Anwendungen eingesetzt wird.


#### Ausblick

Für eine weiterführende Analyse könnten mehrere Erweiterungen sinnvoll sein:

• Verwendung realer Gesundheitsdatensätze  
• Integration zusätzlicher Variablen (z. B. sozioökonomische Faktoren)  
• Untersuchung longitudinaler Daten  
• Anwendung komplexerer Modellverfahren  
• Einsatz kausaler Analyseansätze  

Solche Erweiterungen könnten helfen, die beobachteten Zusammenhänge genauer zu verstehen und mögliche Ursache-Wirkungs-Beziehungen zu untersuchen.


#### Schlussbemerkung

Insgesamt zeigt die Analyse, wie explorative Datenanalyse und maschinelles Lernen gemeinsam eingesetzt werden können, um strukturierte Einblicke in komplexe Datensätze zu gewinnen.

Das Projekt versteht sich daher in erster Linie als methodisch orientierte Data-Science-Analyse, die den vollständigen Weg von der Datenexploration bis zur Modellinterpretation nachvollziehbar demonstriert.

```mermaid
flowchart LR

subgraph N1["Notebook 1"]
A["Daten-<br>Verständnis"]
B["Explorative<br>Datenanalyse"]
end

subgraph N2["Notebook 2"]
C["Feature<br>Engineering"]
D["Strukturanalyse<br/>(PCA / Clustering)"]
E["Modeling<br/>(Regression / Klassifikation)"]
end

subgraph N3["Notebook 3"]
F["Modell-<br>Evaluierung"]
G["Feature<br>Interpretation"]
H["Insights &<br>Reflektion"]
end

A --> B --> C --> D --> E --> F --> G --> H
``` 

![Flowchart](notebook_structure.png)